# 4단계 — 에폭 수 (이 단계는 에폭 수 하나만 바꾼다)

3단계(Dropout 비율) 다음 단계다. 한 번에 하나만 바꾸는 원칙대로 **이번 단계는 에폭 수만 바꾼다.** 조기 종료·데이터 증강은 이 단계가 끝난 뒤 하나씩 따로 본다. 지금까지 모든 실험을 5에폭으로 고정해서 돌렸는데, 5에폭이 알맞은 길이인지는 확인한 적이 없다. 기준선(Adam 1e-3)은 val loss가 2~3에폭에서 바닥을 찍고 올라가서 5에폭이 이미 바닥을 지난 뒤이고, 반대로 학습이 느린 설정(Adam 1e-4 등)은 5에폭에도 바닥에 닿지 못했다. 그래서 학습 곡선을 30에폭까지 보고, 5에폭이 그 곡선의 어디쯤인지 잰다.

| | 값 |
|---|---|
| 프레임워크 | Keras, PyTorch |
| 바꾸는 것 | 에폭 수 5 → **30 고정** (조기 종료 없음). 30에폭을 끝까지 돌린 한 번의 곡선에서 후보 3 · **5(현재 기준선)** · 10 · 20 · 30에폭 끝의 값을 비교한다 |
| Dropout 비율 | 셀 1의 `BASE_DROPOUT` (기본 0.5). 3단계에서 기준선이 옮겨 갔다면 이 한 줄만 바꾼다 |
| 시드 | 0 한 번씩 훑어본다. 시드를 더하려면 셀 1의 `SEEDS`를 `(0, 1, 2)`로 바꾼다 |
| 실행 횟수 | 시드 하나에 Keras 1번 + PyTorch 1번 = **2번** |
| 그 밖의 조건 | 3단계와 같다: He 초기화, 2 Conv(32·64) + Dropout + Dense(11), Adam lr 1e-3, batch 32, 증강 없음, 학습률 고정(스케줄러 없음) |
| 소요 시간 | 대략 45분 (Keras ≈ 16분, PyTorch ≈ 29분). 시드를 `(0, 1, 2)`로 늘리면 시드 0이 끝난 뒤 추가로 약 1시간 30분 |

**Keras·PyTorch 셀은 3단계 노트북의 셀(2단계에서 실제로 돌아간 코드에서 Dropout 줄만 바꾼 것)에서 아래 줄만 바뀌었다.** ① 한 번에 한 설정만 돌리므로 Dropout 비율을 인자로 받지 않고 `BASE_DROPOUT`을 쓴다 ② 후보 에폭이 끝날 때마다 test를 잰다 (PyTorch는 그 자리에서, Keras는 그 에폭의 가중치를 담아 뒀다가 학습이 끝난 뒤에 잰다) ③ 에폭 줄 끝에 지난 시간(분)이 붙는다. 에폭 수는 셀 1의 `EPOCHS`다.

## 에폭 수란

**학습 데이터 전체를 한 번 훑는 것이 1에폭이다.** 에폭 수는 그 훑기를 몇 번 할지 정하는 값이다. 적으면 덜 배운 채 끝나고, 너무 많으면 학습 데이터를 외워서 val 성능이 나빠진다 (과적합).

학습률이 고정이고 스케줄러가 없어서, 30에폭 실행의 앞 k에폭은 k에폭만 돌린 실행과 같은 길을 간다 (GPU 비결정성으로 생기는 작은 차이는 있다). 그래서 후보마다 따로 돌리지 않고 **30에폭을 한 번 돌린 곡선에서 k = 3 · 5 · 10 · 20 · 30에폭 끝의 값을 읽는다.** 가중치를 되돌리거나 중간에 멈추는 장치는 없다. 그건 다음 단계인 조기 종료다.

## 예상되는 작용

- train 정확도는 30에폭까지 계속 오르고, train 대 val 격차는 에폭이 늘수록 벌어진다
- val loss는 2~5에폭에서 바닥을 찍고 그 뒤로 오른다. 과적합이 깊어질수록 모델이 틀린 답을 더 자신 있게 내기 때문이다
- val 정확도는 loss보다 덜 움직인다. 정체하거나 천천히 내려간다
- 학습 시간은 에폭 수에 비례한다 (이 모델은 에폭당 Keras 약 30초, PyTorch 약 55초)

## 이 단계가 바꾸지 않는 것 (한계)

- **조기 종료와 최적 에폭의 자동 선택은 없다.** 에폭 수를 고정해서 돌린 결과만 본다. 바닥에서 멈추는 방법은 다음 단계에서 따로 본다
- **지금 기준선에서만 본다.** 다른 설정이 바뀌면 알맞은 에폭 수도 달라질 수 있다. 정규화를 세게 걸면 학습이 느려져서 더 긴 에폭이 필요할 수 있다. 이 단계의 답은 "현재 기준선의 에폭 수"다
- 후보를 고르는 데 val을 쓴다 (5개 중 고르는 정도라 낙관 편향은 작다). test는 보고용이다

## 결론을 내는 규칙과 예상 (결과를 보기 전에 정해 둔다)

**시드 1개(지금 기본값)에서는 판정하지 않고 곡선과 차이만 본다.** 2·3단계 훑어보기와 같다. 시드 0 한 번의 값은 높게 나오는 경향이 있었고(2단계 3시드 확인에서 Keras가 0.2~2.3%p 높았다), 같은 설정을 다시 돌려도 val이 1~2%p 움직인다. 에폭 수에 따른 변화가 이보다 한참 크지 않으면 읽지 않는다.

**시드 3개를 모으면 다음 규칙으로 판정한다** (2·3단계와 같은 기준이다).

- 후보 에폭 수 하나하나를 기준선(5에폭)과 같은 시드끼리 뺀다. 비교하는 값은 **그 에폭 끝의 val 정확도와 val loss**다 (그 에폭에서 멈춘 모델의 값이고, 1~k에폭 중 최저가 아니다). 시드 전부에서 방향이 같고, 평균 차이가 시드 범위(두 조건의 최대−최소 중 큰 쪽)보다 클 때만 "뚜렷함"이다
- 어떤 후보가 val acc와 val loss **둘 다, 두 프레임워크 모두(4칸 전부)**에서 5에폭보다 뚜렷하게 낫다면 기준선 에폭 수를 그 값으로 옮긴다. 여러 개면 val acc 평균이 가장 높은 것이다. 아니면 5를 유지한다
- 4칸 전부에서 5에폭이 뚜렷하게 낫다면 "기준선이 뚜렷하게 낫다"로 적는다
- train − val 격차와 test 정확도는 보고용이고 판정에 쓰지 않는다
- **val loss 최저 에폭(1~30에폭 중)과 val acc 최고 에폭은 판정이 아니라 곡선의 모양으로 적는다.** 다음 단계(조기 종료)에서 얼마나 기다릴지 정하는 근거다

**재현 확인 (앞 5에폭).** 30에폭 실행의 앞 5에폭은 3단계의 같은 Dropout 비율 실행(같은 시드)과 같은 설정이다. 셀 4가 3단계 CSV(`food11_v2_dropout.csv`)에서 같은 시드의 5에폭 값을 찾아 나란히 보여 준다. 재실행 흔들림(val 1~2%p) 안에서 맞아야 한다. 크게 벗어나면 코드나 환경이 달라진 것이므로 에폭 비교를 읽기 전에 원인부터 확인한다.

### 예상 (가설 — 결과로 확인한다)

1. **train 정확도는 30에폭까지 계속 오르고 train 대 val 격차는 벌어질 것이다.** 5에폭에서 이미 26~36%p였다. 두 프레임워크 모두에서 보일 것으로 본다.
2. **val loss 최저 에폭은 2~5에폭에 있고, 30에폭의 val loss는 5에폭보다 확실히 높을 것이다.** 기준선에서 이미 2~3에폭에 바닥이었다.
3. **val 정확도는 5에폭 근처에서 정체할 것이다.** 10 · 20 · 30에폭이 5에폭보다 뚜렷하게 높지는 않을 것이다 (차이가 시드 범위 안: Keras 2.5~5.1%p, PyTorch 1.1~2.6%p).
4. **3에폭은 5에폭보다 val loss가 낮거나 비슷하지만 val 정확도는 구분되지 않을 것이다.**
5. 그래서 규칙의 결과는 "5 유지"일 가능성이 높다. 이 단계의 가치는 판정보다 곡선의 모양, 특히 val loss 바닥의 위치다.
6. 두 프레임워크에서 같은 방향이 나올 것이다.

## 실행 방법

위에서부터 셀을 차례로 돌린다 (셀 0 → 1 → 2 → 3 → 4). 자리를 비울 때는 셀 0에서 Drive 허용을 누른 뒤 "런타임 → 모두 실행"으로 이어서 돌려도 된다. 끝나면 셀 4가 요약을 낸다.

- **GPU(T4)로 돌린다.** 이전 단계가 전부 T4였다. CPU로도 돌아가지만 몇 시간이 걸리고, GPU와 결과가 조금 달라서 섞으면 비교가 깨끗하지 않다
- 돌리기 전에 셀 1의 `BASE_DROPOUT`이 3단계 결과와 맞는지 본다. 3단계에서 기준선이 안 옮겨 갔다면 0.5 그대로 둔다
- 셀 1의 `[계획]`에 남은 실행 수와 시간 어림이 나온다. 이 단계의 결과는 Drive의 `food11_v2_epochs.csv`에 실행 하나가 끝날 때마다 한 줄씩 쌓인다. 다시 돌리면 끝난 실행은 건너뛴다. 처음부터 다시 하려면 그 파일을 지운다 (3단계의 `food11_v2_dropout.csv`와는 별개 파일이다)
- **실행 하나가 Keras 약 16분, PyTorch 약 29분이다.** 중간에 런타임이 끊기면 그 실행만 처음부터 다시 돈다 (끝난 실행은 남는다)
- 셀 2(Keras)와 셀 3(PyTorch)은 서로 독립이다. 셀 4(요약)는 있는 결과만으로 만든다
- 실행 하나마다 시작 줄 → 에폭 줄 30개 → 끝 줄이 나온다. 에폭 줄 끝의 숫자는 그 실행을 시작하고 지난 분이다
- 시드를 더하려면 셀 1의 `SEEDS`를 `(0, 1, 2)`로 바꾸고 셀 1부터 다시 돌린다. 시드 0은 건너뛰고 1·2만 돈다
- **셀 4의 출력을 통째로 붙여 주면 된다.**

## 어떻게 읽을까

셀 4는 출력과 그래프 2개를 만든다: `[결과표]`(프레임워크마다) · `[재현 확인]` · `[기준선 대비]` · 판정 줄 · 그래프 `epochs_curves.png`(에폭별 곡선)와 `epochs_seeds.png`(후보 에폭별 결과).

- **결과표**의 한 줄은 "그 에폭에서 멈췄다면" 얻는 값이다 (시드가 여러 개면 평균 [최소~최대]). 최저 에폭은 1~k에폭 중 val loss가 가장 낮았던 에폭이다. 비고의 `최저 에폭=마지막`은 그 에폭 끝에서도 val loss가 내려가는 중이었다는 뜻이다 (더 돌려야 할 수 있다). 최저 에폭이 k보다 훨씬 앞이면 이미 바닥을 지난 것이다
- 결과표 밑의 `[곡선 전체]` 줄은 30에폭 전체에서 val loss가 가장 낮았던 에폭과 val acc가 가장 높았던 에폭이다
- **재현 확인**은 앞 5에폭이 3단계 값과 맞는지 먼저 본다
- **기준선 대비**는 시드 1개이면 차이만 나오고 판정은 "판정 안 함"이다. 시드 3개가 되면 위 규칙대로 판정 줄이 나온다
- 곡선 그래프: 파란 선이 val, 회색 점선이 train이다. 연한 선은 시드 하나, 굵은 선은 평균이다. 세로 점선이 후보 에폭이고 실선이 기준선(5)이다. 동그라미는 평균 val loss가 가장 낮은 에폭이다
- 후보 그래프: 점 하나가 시드 하나이고 같은 시드는 같은 가로 위치에 있다 (왼쪽부터 시드 0 · 1 · 2). 짧은 가로선이 평균이다

## 셀 0 — 데이터 복사 (최초 1회, 런타임 끊기면 다시)

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")

# Drive에서 직접 읽으면 학습이 수십 배 느려져서 로컬로 복사한다
if not os.path.isdir("/content/food11"):
    !cp -r /content/drive/MyDrive/food11 /content/food11

## 셀 1 — 공통 설정, 기록 함수, 실행 계획

In [ ]:
import csv
import gc
import json
import os
import time

import numpy as np

DATA_DIR = "/content/food11"
IMG, BATCH = 128, 32
EPOCHS = 30                  # 이 단계에서 바꾸는 것: 5 -> 30 (고정, 조기 종료 없음)
NUM_CLASSES = 11
N_PARAMS = 653_003           # 두 프레임워크 모두 이 값이어야 같은 모델이다

# 이 단계에서 바꾸는 것은 에폭 수 하나다. 나머지(He 초기화, 구조, Adam 1e-3, batch 32,
# 증강 없음, 학습률 고정)는 3단계 그대로다. 조기 종료는 이 단계에서 건드리지 않는다.
OPTIMIZER, LR = "adam", 1e-3
BASE_DROPOUT = 0.5           # 3단계(Dropout) 결과로 기준선이 옮겨 갔다면 이 한 줄만 바꾼다
BASE_EPOCHS = 5              # 현재 기준선
CANDIDATES = (3, 5, 10, 20, 30)      # 비교할 에폭 수 (결과를 보기 전에 정해 둔다). 마지막 값 = EPOCHS
SEEDS = (0,)                 # (0, 1, 2)로 바꾸면 시드 0은 건너뛰고 1·2만 추가로 돈다
MIN_PER_RUN = {"keras": 16, "torch": 29}      # 실행 하나에 걸리는 시간 (분, 어림값)
assert CANDIDATES[-1] == EPOCHS and BASE_EPOCHS in CANDIDATES

# 실행 하나가 끝날 때마다 한 줄씩 쌓는다. 런타임이 끊겨도 남고,
# 다시 돌리면 끝난 실행은 건너뛴다. 처음부터 다시 하려면 이 파일을 지운다.
RESULTS = "/content/drive/MyDrive/food11_v2_epochs.csv"
PREV_RESULTS = "/content/drive/MyDrive/food11_v2_dropout.csv"      # 3단계 결과 (앞 5에폭 재현 확인에 쓴다)
FIELDS = ["framework", "optimizer", "lr", "dropout", "seed", "n_params", "epochs", "minutes",
          "train_acc", "train_loss", "val_acc", "val_loss",    # 에폭별 값 (JSON 리스트, 길이 = EPOCHS)
          "test_acc", "test_loss"]                             # 후보 에폭 끝의 값 (JSON 딕셔너리 {에폭: 값})


def done_runs():
    if not os.path.exists(RESULTS):
        return set()
    with open(RESULTS, newline="", encoding="utf-8") as f:
        return {(r["framework"], float(r["dropout"]), int(r["seed"]))
                for r in csv.DictReader(f)}


def save_row(row):
    is_new = not os.path.exists(RESULTS)
    with open(RESULTS, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        if is_new:
            w.writeheader()
        w.writerow(row)


def make_row(framework, seed, n_params, minutes, hist, test):
    """hist: 에폭별 리스트 4개, test: {후보 에폭: (loss, acc)}"""
    return dict(framework=framework, optimizer=OPTIMIZER, lr=LR, dropout=BASE_DROPOUT, seed=seed,
                n_params=n_params, epochs=EPOCHS, minutes=round(minutes, 2),
                train_acc=json.dumps(hist["train_acc"]),
                train_loss=json.dumps(hist["train_loss"]),
                val_acc=json.dumps(hist["val_acc"]),
                val_loss=json.dumps(hist["val_loss"]),
                test_acc=json.dumps({str(k): v[1] for k, v in test.items()}),
                test_loss=json.dumps({str(k): v[0] for k, v in test.items()}))


def tag(framework, seed):
    return f"[{framework} · {EPOCHS}에폭 · 시드 {seed}]"


def report(row):
    va = json.loads(row["val_acc"])
    vl = np.array(json.loads(row["val_loss"]), dtype=float)
    ok = np.isfinite(vl)
    txt = f"val {va[BASE_EPOCHS - 1]:.3f} ({BASE_EPOCHS}에폭) → {va[-1]:.3f} ({EPOCHS}에폭)"
    if ok.any():
        best = int(np.argmin(np.where(ok, vl, np.inf)))
        txt += f" · val loss 최저 {vl[best]:.2f} (에폭 {best + 1})"
    if not ok[-1]:
        txt += " · 발산(loss NaN)"
    print(f"{tag(row['framework'], row['seed'])} {txt} · {row['minutes']:.1f}분")


def plan():
    done = done_runs()
    print(f"[계획] {EPOCHS}에폭 고정, 후보 {' · '.join(map(str, CANDIDATES))}에폭 (기준선 {BASE_EPOCHS}) · "
          f"Dropout {BASE_DROPOUT} × 시드 {', '.join(map(str, SEEDS))} — CSV에 이미 있는 실행은 건너뛴다")
    for fw, name in (("keras", "Keras"), ("torch", "PyTorch")):
        todo = [s for s in SEEDS if (fw, BASE_DROPOUT, s) not in done]
        print(f"  {name:8s} 이미 {len(SEEDS) - len(todo)}번 · 남은 {len(todo)}번 ≈ {len(todo) * MIN_PER_RUN[fw]:.0f}분")


plan()

## 셀 2 — Keras (30에폭, 시드마다 1번) · 시드 하나에 약 16분

In [ ]:
import keras
from keras import layers


def make_optimizer(name, lr):
    if name == "adam":
        return keras.optimizers.Adam(learning_rate=lr)
    if name == "sgd_momentum":
        return keras.optimizers.SGD(learning_rate=lr, momentum=0.9)
    return keras.optimizers.SGD(learning_rate=lr)


def show_epoch(epoch, logs, t0):
    print(f"  에폭 {epoch + 1}/{EPOCHS}  "
          f"train {logs['accuracy']:.3f} (loss {logs['loss']:.3f})  ·  "
          f"val {logs['val_accuracy']:.3f} (loss {logs['val_loss']:.3f})  ·  "
          f"{(time.time() - t0) / 60:.1f}분", flush=True)


def run_keras(seed):
    keras.utils.set_random_seed(seed)

    def load(split, shuffle):
        ds = keras.utils.image_dataset_from_directory(
            f"{DATA_DIR}/{split}", image_size=(IMG, IMG),
            batch_size=BATCH, shuffle=shuffle, seed=seed)
        rescale = layers.Rescaling(1.0 / 255)
        return ds.map(lambda x, y: (rescale(x), y))

    train_ds = load("training", True)
    val_ds = load("validation", False)
    test_ds = load("evaluation", False)

    # 128 -> conv 126 -> pool 63 -> conv 61 -> pool 30
    model = keras.Sequential([
        keras.Input(shape=(IMG, IMG, 3)),

        layers.Conv2D(32, 3, activation="relu", kernel_initializer="he_normal"),
        layers.MaxPooling2D(2),

        layers.Conv2D(64, 3, activation="relu", kernel_initializer="he_normal"),
        layers.MaxPooling2D(2),

        layers.Flatten(),
        layers.Dropout(BASE_DROPOUT),
        layers.Dense(NUM_CLASSES, activation="softmax",
                     kernel_initializer="he_normal"),
    ])
    n_params = int(sum(np.prod(w.shape) for w in model.trainable_weights))
    assert n_params == N_PARAMS, n_params

    model.compile(
        optimizer=make_optimizer(OPTIMIZER, LR),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    snaps = {}      # 후보 에폭 끝의 가중치. fit 도중에는 evaluate를 부르지 않고, test는 학습이 끝난 뒤 이 가중치로 잰다

    def keep(epoch, logs):
        if epoch + 1 in CANDIDATES:
            snaps[epoch + 1] = model.get_weights()

    t0 = time.time()
    h = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, verbose=0,
                  callbacks=[keras.callbacks.LambdaCallback(
                                 on_epoch_end=lambda e, logs: show_epoch(e, logs, t0)),
                             keras.callbacks.LambdaCallback(on_epoch_end=keep)]).history
    test = {}
    for n, w in sorted(snaps.items()):
        model.set_weights(w)
        loss, acc = model.evaluate(test_ds, verbose=0)
        test[n] = (float(loss), float(acc))
    hist = {"train_acc": h["accuracy"], "train_loss": h["loss"],
            "val_acc": h["val_accuracy"], "val_loss": h["val_loss"]}
    row = make_row("keras", seed, n_params, (time.time() - t0) / 60,
                   {k: [float(v) for v in vs] for k, vs in hist.items()}, test)
    keras.backend.clear_session()
    gc.collect()
    return row


for seed in SEEDS:
    if ("keras", BASE_DROPOUT, seed) in done_runs():
        print(f"{tag('keras', seed)} 이미 있음 — 건너뜀")
        continue
    print(f"{tag('keras', seed)} 시작 (약 {MIN_PER_RUN['keras']}분)", flush=True)
    row = run_keras(seed)
    save_row(row)
    report(row)

## 셀 3 — PyTorch (30에폭, 시드마다 1번) · 시드 하나에 약 29분

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")

tf_ = transforms.Compose([
    transforms.Resize((IMG, IMG)),
    transforms.ToTensor(),          # 0~1 변환 + HWC -> CHW
])


def make_optimizer(name, params, lr):
    if name == "adam":
        return torch.optim.Adam(params, lr=lr)
    if name == "sgd_momentum":
        return torch.optim.SGD(params, lr=lr, momentum=0.9)
    return torch.optim.SGD(params, lr=lr)


def run_torch(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    g = torch.Generator().manual_seed(seed)

    def load(split, shuffle):
        ds = datasets.ImageFolder(f"{DATA_DIR}/{split}", transform=tf_)
        return DataLoader(ds, batch_size=BATCH, shuffle=shuffle,
                          generator=g if shuffle else None,
                          num_workers=2, pin_memory=True)

    train_dl = load("training", True)
    val_dl = load("validation", False)
    test_dl = load("evaluation", False)

    # 128 -> conv 126 -> pool 63 -> conv 61 -> pool 30
    model = nn.Sequential(
        nn.Conv2d(3, 32, 3), nn.ReLU(), nn.MaxPool2d(2, 2),
        nn.Conv2d(32, 64, 3), nn.ReLU(), nn.MaxPool2d(2, 2),
        nn.Flatten(),
        nn.Dropout(BASE_DROPOUT),
        nn.Linear(64 * 30 * 30, NUM_CLASSES),
    ).to(device)

    for m in model:       # 1단계 — He 초기화. 가중치만, bias는 PyTorch 기본값 그대로
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            nn.init.kaiming_normal_(m.weight, mode="fan_in", nonlinearity="relu")

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    assert n_params == N_PARAMS, n_params

    loss_fn = nn.CrossEntropyLoss()
    optimizer = make_optimizer(OPTIMIZER, model.parameters(), LR)

    def evaluate(dataloader):
        model.eval()
        total, correct, n = 0.0, 0, 0
        with torch.no_grad():
            for X, y in dataloader:
                X, y = X.to(device), y.to(device)
                pred = model(X)
                total += loss_fn(pred, y).item() * y.size(0)
                correct += (pred.argmax(1) == y).sum().item()
                n += y.size(0)
        return total / n, correct / n

    hist = {"train_acc": [], "train_loss": [], "val_acc": [], "val_loss": []}
    test = {}           # 후보 에폭 끝의 test (loss, acc) — 보고용
    t0 = time.time()
    for epoch in range(EPOCHS):
        model.train()
        total, correct, n = 0.0, 0, 0
        for X, y in train_dl:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            loss = loss_fn(pred, y)
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            total += loss.item() * y.size(0)
            correct += (pred.argmax(1) == y).sum().item()
            n += y.size(0)
        val_loss, val_acc = evaluate(val_dl)
        hist["train_acc"].append(correct / n)
        hist["train_loss"].append(total / n)
        hist["val_acc"].append(val_acc)
        hist["val_loss"].append(val_loss)
        print(f"  에폭 {epoch + 1}/{EPOCHS}  "
              f"train {hist['train_acc'][-1]:.3f} (loss {hist['train_loss'][-1]:.3f})  ·  "
              f"val {val_acc:.3f} (loss {val_loss:.3f})  ·  "
              f"{(time.time() - t0) / 60:.1f}분", flush=True)
        if epoch + 1 in CANDIDATES:
            test[epoch + 1] = evaluate(test_dl)

    row = make_row("torch", seed, n_params, (time.time() - t0) / 60, hist, test)
    del model, optimizer, train_dl, val_dl, test_dl
    torch.cuda.empty_cache()
    gc.collect()
    return row


for seed in SEEDS:
    if ("torch", BASE_DROPOUT, seed) in done_runs():
        print(f"{tag('torch', seed)} 이미 있음 — 건너뜀")
        continue
    print(f"{tag('torch', seed)} 시작 (약 {MIN_PER_RUN['torch']}분)", flush=True)
    row = run_torch(seed)
    save_row(row)
    report(row)

## 셀 4 — 요약과 판정 (이 셀의 출력을 붙여 주면 된다)

In [ ]:
import unicodedata
import warnings

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

with open(RESULTS, newline="", encoding="utf-8") as f:
    rows = [r for r in csv.DictReader(f) if float(r["dropout"]) == BASE_DROPOUT]    # 다른 Dropout 비율의 예전 결과는 쓰지 않는다
assert rows, f"Dropout {BASE_DROPOUT}의 결과가 아직 없다 — 셀 2·3을 먼저 돌릴 것"
assert all(int(r["n_params"]) == N_PARAMS for r in rows), "파라미터 수가 다른 실행이 있다"
assert all(len(json.loads(r["val_loss"])) == EPOCHS for r in rows), f"{EPOCHS}에폭이 아닌 실행이 있다"

FW = {"keras": "Keras", "torch": "PyTorch"}
MIN_SEEDS = 3                  # 규칙을 적용하려면 조건마다 이만큼의 시드가 필요하다


def width(s):
    return sum(2 if unicodedata.east_asian_width(c) in "WF" else 1 for c in s)


def pad(s, w):
    return s + " " * max(0, w - width(s))


def bar(title):
    print()
    print("=" * 100)
    print(title)
    print("=" * 100)


def metrics(r):
    """한 실행에서 후보 에폭마다의 값(at), 30에폭 전체의 모양(whole), 에폭별 곡선(curve)을 뽑는다."""
    ta, tl = np.array(json.loads(r["train_acc"])), np.array(json.loads(r["train_loss"]))
    va, vl = np.array(json.loads(r["val_acc"])), np.array(json.loads(r["val_loss"]))
    t_acc = json.loads(r["test_acc"])
    at = {}
    for k in CANDIDATES:
        head = vl[:k]
        ok = np.isfinite(head)
        best = int(np.argmin(np.where(ok, head, np.inf)))
        at[k] = dict(train_acc=ta[k - 1] * 100, val_acc=va[k - 1] * 100, gap=(ta[k - 1] - va[k - 1]) * 100,
                     val_loss=float(vl[k - 1]), best_ep=best + 1 if ok.any() else np.nan,
                     test_acc=float(t_acc[str(k)]) * 100,
                     diverged=bool(not np.isfinite(vl[k - 1]) or not np.isfinite(tl[k - 1])))
    ok = np.isfinite(vl)
    best = int(np.argmin(np.where(ok, vl, np.inf)))
    top = int(np.argmax(np.where(np.isfinite(va), va, -np.inf)))
    whole = dict(vl_min=float(vl[best]) if ok.any() else np.nan, vl_min_ep=best + 1 if ok.any() else np.nan,
                 va_max=float(va[top]) * 100, va_max_ep=top + 1)
    curve = dict(train_acc=ta * 100, val_acc=va * 100, train_loss=tl, val_loss=vl)
    return at, whole, curve


runs = {}        # (프레임워크, 에폭 수) -> {시드: 지표}
whole_by = {}    # 프레임워크 -> {시드: 30에폭 전체의 모양}
curves = {}      # 프레임워크 -> {시드: 에폭별 곡선}
for r in rows:
    fw, s = r["framework"], int(r["seed"])
    at, whole, curve = metrics(r)
    for k, m in at.items():
        runs.setdefault((fw, k), {})[s] = m
    whole_by.setdefault(fw, {})[s] = whole
    curves.setdefault(fw, {})[s] = curve

fws = [fw for fw in FW if fw in curves]
assert fws, "결과가 CSV에 없다 — 셀 1의 [계획]을 확인할 것"
seeds_all = sorted({s for fw in fws for s in curves[fw]})


def vals(group, key):
    v = np.array([m[key] for m in group.values()], dtype=float)
    return v[np.isfinite(v)]


def span(group, key, nd):
    v = vals(group, key)
    if len(v) == 0:
        return "-"
    if key.endswith("_ep") and len(group) == 1:
        nd = 0
    if len(group) == 1:
        return f"{v[0]:.{nd}f}"
    return f"{v.mean():.{nd}f} [{v.min():.{nd}f}~{v.max():.{nd}f}]"


def mean_of(group, key):
    v = vals(group, key)
    return float(v.mean()) if len(v) else np.nan


def notes(k, group):
    out = []
    if k == BASE_EPOCHS:
        out.append("기준선")
    n_div = sum(m["diverged"] for m in group.values())
    if n_div:
        out.append(f"발산 {n_div}/{len(group)}")
    live = [m for m in group.values() if not m["diverged"]]
    n_last = sum(m["best_ep"] == k for m in live)
    if live and n_last == len(live):
        out.append("최저 에폭=마지막")
    elif n_last:
        out.append(f"최저 에폭=마지막 {n_last}/{len(live)}")
    return " · ".join(out)


# ---- 결과표 -----------------------------------------------------------------
COLS = [("train_acc", "train acc %", 1), ("val_acc", "val acc %", 1), ("gap", "train − val %p", 1),
        ("val_loss", "val loss", 2), ("best_ep", "최저 에폭", 1), ("test_acc", "test acc %", 1)]
for fw in fws:
    bar(f"[결과표] {FW[fw]} — Dropout {BASE_DROPOUT}, Adam 1e-3, 시드 {', '.join(map(str, sorted(curves[fw])))}"
        + ("" if len(curves[fw]) == 1 else ", 평균 [최소~최대]"))
    print("   한 줄은 '그 에폭에서 멈췄다면' 얻는 값이다. 최저 에폭은 1~k에폭 중 val loss가 가장 낮았던 에폭이다")
    head = ["에폭 수", "시드"] + [name for _, name, _ in COLS] + ["비고"]
    body = []
    for k in CANDIDATES:
        g = runs[(fw, k)]
        body.append([str(k), ",".join(map(str, sorted(g)))]
                    + [span(g, key, nd) for key, _, nd in COLS] + [notes(k, g)])
    ws = [max(width(r[i]) for r in [head] + body) + 2 for i in range(len(head) - 1)]
    for r in [head] + body:
        print("".join(pad(c, w) for c, w in zip(r[:-1], ws)) + r[-1])
    w_ = whole_by[fw]
    print(f"[곡선 전체] 1~{EPOCHS}에폭 중 val loss 최저 {span(w_, 'vl_min', 2)} (에폭 {span(w_, 'vl_min_ep', 1)}) · "
          f"val acc 최고 {span(w_, 'va_max', 1)}% (에폭 {span(w_, 'va_max_ep', 1)})")


# ---- 앞 5에폭 재현 확인 -----------------------------------------------------
def prev_row(fw, seed):
    with open(PREV_RESULTS, newline="", encoding="utf-8") as f:
        for r in csv.DictReader(f):
            if (r["framework"], float(r["dropout"]), int(r["seed"])) == (fw, BASE_DROPOUT, seed):
                return r
    return None


bar(f"[재현 확인] 앞 {BASE_EPOCHS}에폭은 3단계 Dropout {BASE_DROPOUT} 실행(같은 시드)과 같은 설정이다 — "
    "재실행 흔들림(val 1~2%p) 안에서 맞아야 한다")
if not os.path.exists(PREV_RESULTS):
    print(f"   3단계 결과 파일({PREV_RESULTS})이 없어 비교하지 않는다 — 3단계 결과표의 Dropout {BASE_DROPOUT} 행과 직접 비교할 것")
else:
    e = BASE_EPOCHS - 1
    for fw in fws:
        for s in sorted(curves[fw]):
            pr = prev_row(fw, s)
            if pr is None or len(json.loads(pr["val_acc"])) <= e:
                print(f"   {FW[fw]:8s} 시드 {s}: 3단계에 Dropout {BASE_DROPOUT} 시드 {s} 실행이 없다")
                continue
            now = curves[fw][s]
            p_ta, p_va = json.loads(pr["train_acc"])[e] * 100, json.loads(pr["val_acc"])[e] * 100
            p_vl = json.loads(pr["val_loss"])[e]
            print(f"   {pad(FW[fw], 9)}시드 {s}  "
                  f"train acc {now['train_acc'][e]:.1f} (3단계 {p_ta:.1f}, {now['train_acc'][e] - p_ta:+.1f}%p) · "
                  f"val acc {now['val_acc'][e]:.1f} (3단계 {p_va:.1f}, {now['val_acc'][e] - p_va:+.1f}%p) · "
                  f"val loss {now['val_loss'][e]:.2f} (3단계 {p_vl:.2f}, {now['val_loss'][e] - p_vl:+.2f})")


# ---- 같은 시드끼리 비교 -----------------------------------------------------
def paired(fw, a, b, key, higher_better):
    """같은 시드끼리 뺀 값 a-b로 a에폭과 b에폭을 비교한다. higher_better=False면 값이 낮은 쪽이 낫다."""
    ga, gb = runs.get((fw, a), {}), runs.get((fw, b), {})
    seeds = [s for s in sorted(set(ga) & set(gb))
             if np.isfinite(ga[s][key]) and np.isfinite(gb[s][key])]
    out = dict(diffs=[], mean=np.nan, limit=np.nan, winner=None, enough=len(seeds) >= MIN_SEEDS,
               text="비교할 공통 시드가 없다")
    if not seeds:
        return out
    va = np.array([ga[s][key] for s in seeds])
    vb = np.array([gb[s][key] for s in seeds])
    d = va - vb
    out.update(diffs=[float(x) for x in d], mean=float(d.mean()))
    if not out["enough"]:
        out["text"] = f"판정 안 함 (공통 시드 {len(seeds)}개, {MIN_SEEDS}개 필요)"
        return out
    gain = d if higher_better else -d              # 양수 = a가 낫다
    limit = float(max(np.ptp(va), np.ptp(vb)))      # 시드 범위: 두 조건 중 큰 쪽
    out["limit"] = limit
    consistent = bool((gain > 0).all() or (gain < 0).all())
    if consistent and abs(float(gain.mean())) > limit:
        winner = "a" if gain.mean() > 0 else "b"
        out["winner"] = winner
        out["text"] = f"{a if winner == 'a' else b}에폭 우세 (뚜렷함)"
    elif not consistent:
        out["text"] = (f"구분 안 됨 — 시드마다 방향이 다름 "
                       f"({a}에폭 우세 {int((gain > 0).sum())}/{len(seeds)})")
    else:
        out["text"] = "구분 안 됨 — 방향은 같지만 평균 차이가 시드 범위 안"
    return out


def show_cmp(fw, a, b, key, higher_better, title, nd, unit, judge=True):
    c = paired(fw, a, b, key, higher_better)
    head = pad(FW[fw], 9) + pad(f"{a}에폭", 8) + pad(title, 18)
    if not c["diffs"]:
        print(head + c["text"])
        return c
    per = " ".join(f"{x:+.{nd}f}" for x in c["diffs"])
    lim = f"  시드 범위 {c['limit']:.{nd}f}" if c["enough"] else ""
    tail = f"  -> {c['text']}" if judge else "  (보고용)"
    print(head + f"{c['mean']:+.{nd}f}{unit}  (시드별 {per}){lim}" + tail)
    return c


bar(f"[기준선 대비] 같은 시드끼리 뺀 값 (k에폭 끝 − 기준선 {BASE_EPOCHS}에폭 끝), val loss는 낮은 쪽이 낫다\n"
    "   뚜렷함 = 시드 전부 같은 방향이고 평균 차이가 시드 범위(두 조건의 최대−최소 중 큰 쪽)보다 클 것\n"
    "   시드가 3개 미만이면 판정하지 않고 차이만 보여 준다 (훑어보기)")
cells = {}
for k in CANDIDATES:
    if k == BASE_EPOCHS:
        continue
    cells[k] = []
    for fw in fws:
        cells[k].append(show_cmp(fw, k, BASE_EPOCHS, "val_acc", True, "val acc", 1, "%p"))
        cells[k].append(show_cmp(fw, k, BASE_EPOCHS, "val_loss", False, "val loss", 3, ""))
        show_cmp(fw, k, BASE_EPOCHS, "gap", False, "train − val 격차", 1, "%p", judge=False)
        show_cmp(fw, k, BASE_EPOCHS, "test_acc", True, "test acc", 1, "%p", judge=False)
    print()

print(f"-> 판정 (4칸 = val acc · val loss × 두 프레임워크, 기준선 {BASE_EPOCHS}에폭 대비)")
movers = []
all_hold = True
for k, cs in cells.items():
    n_new = sum(c["winner"] == "a" for c in cs)
    n_base = sum(c["winner"] == "b" for c in cs)
    n_skip = sum(not c["enough"] for c in cs)
    n_tie = len(cs) - n_new - n_base - n_skip
    if len(fws) < 2 or n_skip:
        label = "판정 보류"
    else:
        all_hold = False
        if n_new == len(cs):
            label = "기준선보다 뚜렷하게 낫다 -> 기준선 후보"
            movers.append(k)
        elif n_base == len(cs):
            label = "기준선이 뚜렷하게 낫다"
        else:
            label = "구분 안 됨"
    extra = f" · 판정 안 함 {n_skip}칸" if n_skip else ""
    print(f"   {k}에폭: {label}  ({k}에폭 우세 {n_new}칸 · 기준선 우세 {n_base}칸 · 구분 안 됨 {n_tie}칸{extra})")
print()
if all_hold:
    print("-> 결론: 판정 보류 — 두 프레임워크 모두 시드 3개씩이 필요하다. 지금은 훑어보기이므로 곡선의 모양만 읽는다")
elif movers:
    def score(k):
        return np.mean([mean_of(runs[(fw, k)], "val_acc") for fw in fws if (fw, k) in runs])
    pick = max(movers, key=score)
    print(f"-> 결론: 기준선 에폭 수를 {BASE_EPOCHS}에서 {pick}으로 옮긴다 (4칸 전부에서 {BASE_EPOCHS}에폭보다 뚜렷하게 앞섰다"
          + (f", 후보 {', '.join(str(m) for m in movers)} 중 val acc 평균이 가장 높다)" if len(movers) > 1 else ")"))
else:
    print(f"-> 결론: 에폭 수 {BASE_EPOCHS} 유지 — 어느 후보도 4칸 전부에서 기준선보다 뚜렷하게 앞서지 않았다")

# ---- 그래프 -----------------------------------------------------------------
SURFACE, INK, INK2, MUTED, GRID, AXIS = ("#fcfcfb", "#0b0b0b", "#52514e",
                                         "#898781", "#e1e0d9", "#c3c2b7")
BLUE = "#2a78d6"
plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.unicode_minus"] = False


def style(ax, grid_axis="y"):
    ax.set_facecolor(SURFACE)
    ax.grid(True, axis=grid_axis, color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(AXIS)
    ax.tick_params(colors=MUTED, labelsize=9)


# 그래프 1 — 에폭별 곡선 (파랑 = val, 회색 점선 = train, 연한 선 = 시드 하나, 굵은 선 = 평균)
ep = np.arange(1, EPOCHS + 1)
PAIRS = [("accuracy (%)", "train_acc", "val_acc"), ("loss", "train_loss", "val_loss")]
fig, axes = plt.subplots(len(fws), 2, figsize=(11, 3.8 * len(fws) + 1.0), dpi=150,
                         squeeze=False, facecolor=SURFACE)
for i, fw in enumerate(fws):
    for j, (name, tr_key, va_key) in enumerate(PAIRS):
        ax = axes[i][j]
        style(ax, "y")
        for k in CANDIDATES:
            ax.axvline(k, color=AXIS, lw=1.8 if k == BASE_EPOCHS else 0.8,
                       ls="-" if k == BASE_EPOCHS else ":", zorder=1)
        for key, color, ls, lw in ((tr_key, MUTED, "--", 2.0), (va_key, BLUE, "-", 2.6)):
            arr = np.array([c[key] for c in curves[fw].values()], dtype=float)      # (시드, 에폭)
            if len(arr) > 1:
                for one in arr:
                    ax.plot(ep, one, color=color, lw=0.9, alpha=0.3, zorder=2)
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", RuntimeWarning)
                mean = np.nanmean(arr, axis=0)
            ax.plot(ep, mean, color=color, lw=lw, ls=ls, zorder=3)
            if key == "val_loss" and np.isfinite(mean).any():
                low = int(np.nanargmin(mean))
                ax.plot([ep[low]], [mean[low]], ls="none", marker="o", ms=8, mfc=BLUE,
                        mec=SURFACE, mew=1.4, zorder=5)
        ax.set_xlim(0.5, EPOCHS + 0.5)
        ax.set_xticks(sorted(set((1,) + CANDIDATES)))
        ax.set_xlabel("epoch", fontsize=9, color=INK2)
        ax.set_title(f"{FW[fw]} · {name}", fontsize=10.5, color=INK, loc="left")
handles = [Line2D([0], [0], color=BLUE, lw=2.6, label="val"),
           Line2D([0], [0], color=MUTED, lw=2.0, ls="--", label="train")]
if len(seeds_all) > 1:
    handles.append(Line2D([0], [0], color=BLUE, lw=0.9, alpha=0.3, label="each seed (thin) / mean (thick)"))
handles.append(Line2D([0], [0], marker="o", ls="none", ms=8, mfc=BLUE, mec=SURFACE, mew=1.4,
                      label="lowest mean val loss"))
fig.text(0.005, 0.99, f"Learning curves, dropout {BASE_DROPOUT}, Adam 1e-3  -  "
         f"vertical lines = candidate epoch counts (solid = baseline {BASE_EPOCHS})",
         fontsize=12, color=INK, va="top")
fig.legend(handles=handles, loc="upper left", bbox_to_anchor=(0.0, 0.955),
           ncol=len(handles), frameon=False, fontsize=9, labelcolor=INK2)
fig.tight_layout(rect=(0, 0, 1, 0.9))
fig.savefig("epochs_curves.png", facecolor=SURFACE)
plt.show()
print("[그래프] epochs_curves.png")

# 그래프 2 — 후보 에폭별 결과 (점 하나 = 시드 하나)
PANELS = [("val_acc", "val acc (%) at k epochs", 1.0),
          ("val_loss", "val loss at k epochs", 0.03),
          ("gap", "train - val acc gap (%p) at k epochs", 1.0)]
fig, axes = plt.subplots(len(fws), 3, figsize=(13, 3.6 * len(fws) + 0.9), dpi=150,
                         squeeze=False, facecolor=SURFACE)
for i, fw in enumerate(fws):
    for j, (key, name, min_pad) in enumerate(PANELS):
        ax = axes[i][j]
        style(ax)
        seen = []
        for x, k in enumerate(CANDIDATES):
            pts = [(s, m[key]) for s, m in sorted(runs.get((fw, k), {}).items()) if np.isfinite(m[key])]
            if not pts:
                continue
            ys = np.array([p[1] for p in pts])
            seen += list(ys)
            ax.vlines(x, ys.min(), ys.max(), color=BLUE, lw=1.6, alpha=0.45, zorder=2)
            ax.plot([x - 0.3, x + 0.3], [ys.mean()] * 2, color=BLUE, lw=2.4,
                    solid_capstyle="butt", zorder=3)
            ax.plot([x + 0.2 * (s - 1) for s, _ in pts], ys, ls="none", marker="o", ms=6.5,
                    mfc=BLUE, mec=SURFACE, mew=1.1, zorder=4)
        if seen:
            lo, hi = min(seen), max(seen)
            margin = max(min_pad, (hi - lo) * 0.2)
            ax.set_ylim(lo - margin, hi + margin)
        ax.set_xlim(-0.6, len(CANDIDATES) - 0.4)
        ax.set_xticks(range(len(CANDIDATES)))
        ax.set_xticklabels([str(k) + (" (base)" if k == BASE_EPOCHS else "") for k in CANDIDATES])
        ax.set_xlabel("epochs", fontsize=9, color=INK2)
        ax.set_title(f"{FW[fw]} · {name}", fontsize=10.5, color=INK, loc="left")
handles = [Line2D([0], [0], color=BLUE, lw=2.4, label="mean"),
           Line2D([0], [0], marker="o", ls="none", ms=6.5, mfc=BLUE, mec=SURFACE, mew=1.1,
                  label="each seed (left to right: 0, 1, 2)")]
fig.text(0.005, 0.99, f"Epoch count (value at the end of k epochs), dropout {BASE_DROPOUT}, Adam 1e-3, He init",
         fontsize=12, color=INK, va="top")
fig.legend(handles=handles, loc="upper left", bbox_to_anchor=(0.0, 0.955),
           ncol=len(handles), frameon=False, fontsize=9, labelcolor=INK2)
fig.tight_layout(rect=(0, 0, 1, 0.9))
fig.savefig("epochs_seeds.png", facecolor=SURFACE)
plt.show()
print("[그래프] epochs_seeds.png")